In [3]:
import pandas as pd


imoex_df=pd.read_csv("../data/raw/imoex_daily.csv")
prices_df=pd.read_csv("../data/raw/prices_daily.csv")


In [4]:
uni_df=pd.read_csv("../data/raw/moex_sectors_history.csv")
sector_to_ticker_df=pd.read_csv("../config/universe.csv")

In [5]:
sector_to_ticker_df.rename({'secid':	'ticker'},axis=1,inplace=True)
sector_to_ticker_df.head()

,ticker,shortname,boardid,sector,is_pref,snapshot_date
0,AFKS,Система ао,TQBR,финансы,0,2026-11-02
1,AFLT,Аэрофлот,TQBR,транспорт,0,2026-11-02
2,ALRS,АЛРОСА ао,TQBR,металлы и добыча,0,2026-11-02
3,BSPB,БСП ао,TQBR,финансы,0,2026-11-02
4,CBOM,МКБ ао,TQBR,финансы,0,2026-11-02


In [6]:
sector_to_ticker_df.columns=['ticker',	'shortname','boardid',	'sector',	'is_pref',	'snapshot_date']

In [7]:
uni_df.head()

,SECID,TRADEDATE,OPEN,HIGH,LOW,CLOSE,VALUE,SECTOR_NAME
0,MOEXOG,2020-01-03,8789.86,8883.83,8788.59,8853.65,2.209356e+10,Нефть и газ
1,MOEXOG,2020-01-06,8872.88,8878.20,8827.28,8875.51,2.192850e+10,Нефть и газ
2,MOEXOG,2020-01-08,8892.73,8995.44,8871.71,8963.74,2.450956e+10,Нефть и газ
3,MOEXOG,2020-01-09,8946.38,9051.84,8913.40,9039.14,2.455164e+10,Нефть и газ
4,MOEXOG,2020-01-10,9023.92,9080.36,8985.64,9037.75,2.106035e+10,Нефть и газ


In [8]:
uni_df.columns=['secid', 'trade_date', 'open_uv', 'high_uv', 'low_uv', 'close_uv', 'value_uv','sector']
uni_df.set_index('trade_date',inplace=True)


In [9]:
uni_df['sector']=uni_df['sector'].apply(lambda x: x.lower())


In [10]:
imoex_df.rename(columns={"close":"close_ix"},inplace=True)
imoex_df.drop("index_code", axis=1, inplace=True)
imoex_df.set_index('trade_date',inplace=True)

In [11]:
prices_df.rename(columns={"secid":"ticker"},inplace=True)
prices_df.set_index('trade_date',inplace=True)

In [12]:
# prices_ix_df.to_csv('../data/processed/prices_clean_no_secs.csv')

In [13]:
prices_ix_df=pd.read_csv("../data/processed/prices_clean_no_secs.csv")

In [14]:
# Убеждаемся, что trade_date доступен как колонка для объединения
if 'trade_date' not in uni_df.columns:
    uni_df = uni_df.reset_index()

if 'trade_date' not in prices_ix_df.columns:
    prices_ix_df = prices_ix_df.reset_index()

# Маппинг тикеров к секторам (все тикеры из universe)
sector_to_ticker_map = sector_to_ticker_df[['ticker', 'sector']].drop_duplicates()
sector_to_ticker_map['sector'] = sector_to_ticker_map['sector'].str.lower().str.strip()
uni_df['sector'] = uni_df['sector'].str.lower().str.strip()


In [15]:
# Привязываем отраслевые индексы ко всем входящим в них тикерам
uni_df_with_tickers = pd.merge(uni_df, sector_to_ticker_map, on='sector', how='inner')
uni_df_with_tickers.head()


,trade_date,secid,open_uv,high_uv,low_uv,close_uv,value_uv,sector,ticker
0,2020-01-03,MOEXOG,8789.86,8883.83,8788.59,8853.65,2.209356e+10,нефть и газ,GAZP
1,2020-01-03,MOEXOG,8789.86,8883.83,8788.59,8853.65,2.209356e+10,нефть и газ,LKOH
2,2020-01-03,MOEXOG,8789.86,8883.83,8788.59,8853.65,2.209356e+10,нефть и газ,NVTK
3,2020-01-03,MOEXOG,8789.86,8883.83,8788.59,8853.65,2.209356e+10,нефть и газ,ROSN
4,2020-01-03,MOEXOG,8789.86,8883.83,8788.59,8853.65,2.209356e+10,нефть и газ,SNGS


In [19]:
# Объединяем цены акций и индекс IMOEX с отраслевыми показателями по тикеру и дате
prices_ix_secs_df = pd.merge(
    prices_ix_df,
    uni_df_with_tickers[['ticker', 'trade_date', 'sector', 'secid', 'open_uv', 'close_uv', 'low_uv', 'high_uv', 'value_uv']],
    on=['ticker', 'trade_date'],
    how='left'  # 'inner' оставляет даты с 2020 г., где есть отраслевые индексы; 'left' сохранит все даты prices_ix_df
)

# Устанавливаем trade_date как индекс
prices_ix_secs_df.set_index('trade_date', inplace=True)

print('Final dataset shape:', prices_ix_secs_df.shape)
prices_ix_secs_df.head()


Final dataset shape: (76843, 17)


,close_ix,boardid,ticker,open,high,low,close,volume,value,numtrades,sector,secid,open_uv,close_uv,low_uv,high_uv,value_uv
trade_date,,,,,,,,,,,,,,,,,
2018-01-03,2152.97,TQBR,AFKS,12.485,12.55,12.26,12.350,20687500,256119052.0,5071,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,2152.97,TQBR,AFLT,139.100,140.95,138.60,140.400,2126700,296982430.0,3163,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,2152.97,TQBR,ALRS,75.210,75.96,75.15,75.810,3908500,295337171.0,3719,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,2152.97,TQBR,BSPB,54.800,54.85,54.55,54.850,8730,477857.5,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,2152.97,TQBR,CBOM,4.680,4.74,4.68,4.735,1065500,5039994.8,294,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
prices_ix_secs_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 76843 entries, 2018-01-03 to 2026-11-02
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   close_ix   76843 non-null  float64
 1   boardid    76843 non-null  object 
 2   ticker     76843 non-null  object 
 3   open       76605 non-null  float64
 4   high       76605 non-null  float64
 5   low        76605 non-null  float64
 6   close      76605 non-null  float64
 7   volume     76843 non-null  int64  
 8   value      76843 non-null  float64
 9   numtrades  76843 non-null  int64  
 10  sector     56722 non-null  object 
 11  secid      56722 non-null  object 
 12  open_uv    56722 non-null  float64
 13  close_uv   56722 non-null  float64
 14  low_uv     56722 non-null  float64
 15  high_uv    56722 non-null  float64
 16  value_uv   56722 non-null  float64
dtypes: float64(11), int64(2), object(4)
memory usage: 10.6+ MB


In [21]:
prices_ix_secs_df.to_csv("../data/processed/prices_ix_secs.csv")